# **2. PyTorch로 CNN 만들기**


---





- 목표
    - ① SmallCNN을 직접 정의하고 층별 shape 추적
    - ② 같은 조건에서 MLP와 공정 비교
    - ③ 학습된 필터·특징맵 시각화
    - ④ 혼동행렬로 약점 분석 |
- 데이터
    - Fashion-MNIST (Xiao et al., 2017, arXiv:1708.07747)
    — 28×28 흑백 의류 10클래스, 학습 60,000 / 테스트 10,000 |
- 실행 환경
    - Colab 또는 PC, CPU / GPU





---



**(코랩에서)한글 폰트 지정하는 방법**

In [ ]:
# 코랩에서 실행
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import shutil
import warnings
import logging

# ── 1. 나눔 폰트 설치 ──────────────────────────────────────────
!apt-get install -y fonts-nanum > /dev/null 2>&1

# ── 2. matplotlib 캐시 디렉터리 삭제 ──────────────────────────
shutil.rmtree(mpl.get_cachedir(), ignore_errors=True)

# ── 3. 폰트 등록 및 유니코드 마이너스 경고 방지 ──────────────────
font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()
plt.rcParams['font.family'] = font_name
plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (6, 4)

# 수학 기호(마이너스 등)를 일반 텍스트로 처리하도록 설정하여 폰트 에러 차단
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.default'] = 'regular'

# mathtext 및 font_manager 관련 폰트 결함 경고 완전히 무시 설정
logging.getLogger('matplotlib.mathtext').setLevel(logging.ERROR)
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib.font_manager')

## 0. 환경 설정

In [ ]:
# [Colab 명령] 한글 폰트: !apt-get -qq install -y fonts-nanum > /dev/null ; 이후 세션 다시 시작
import os, time, subprocess
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

def set_korean_font():
    installed = {f.name for f in fm.fontManager.ttflist}
    for name in ["Malgun Gothic", "NanumGothic", "AppleGothic", "Noto Sans CJK KR"]:
        if name in installed:
            plt.rcParams["font.family"] = name; break
    plt.rcParams["axes.unicode_minus"] = False

set_korean_font()
torch.manual_seed(42); np.random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(max(1, os.cpu_count() or 1))
print("PyTorch", torch.__version__, "| device =", device, "| CPU 코어 =", os.cpu_count())



---



## **2-1. 데이터 준비** — Fashion-MNIST

기본은 `torchvision`이 자동으로 내려받습니다.

In [ ]:
ROOT = "./data"
tfm = transforms.ToTensor()   # [0,255] uint8 → [0,1] float, (1,28,28)

def load_fashion_mnist(root=ROOT):
    try:
        tr = datasets.FashionMNIST(root, train=True, download=True, transform=tfm)
        te = datasets.FashionMNIST(root, train=False, download=True, transform=tfm)
        return tr, te
    except Exception as e:
        print("기본 다운로드 실패 → GitHub 저장소에서 받습니다:", type(e).__name__)
        raw = os.path.join(root, "FashionMNIST", "raw"); os.makedirs(raw, exist_ok=True)
        tmp = os.path.join(root, "_fm_repo")
        if not os.path.exists(tmp):
            subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse",
                            "https://github.com/zalandoresearch/fashion-mnist", tmp], check=True)
            subprocess.run(["git", "-C", tmp, "sparse-checkout", "set", "data/fashion"], check=True)
        import gzip, shutil
        for f in os.listdir(os.path.join(tmp, "data", "fashion")):
            with gzip.open(os.path.join(tmp, "data", "fashion", f)) as src, \
                 open(os.path.join(raw, f[:-3]), "wb") as dst:
                shutil.copyfileobj(src, dst)
        return (datasets.FashionMNIST(root, train=True, download=False, transform=tfm),
                datasets.FashionMNIST(root, train=False, download=False, transform=tfm))

train_full, test_ds = load_fashion_mnist()
train_ds, val_ds = random_split(train_full, [54000, 6000], generator=torch.Generator().manual_seed(42))
CLASSES = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Boot"]
print(f"학습 {len(train_ds):,} / 검증 {len(val_ds):,} / 테스트 {len(test_ds):,}")

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=512)
test_loader  = DataLoader(test_ds, batch_size=512)

fig, axes = plt.subplots(2, 10, figsize=(14, 3.2))
for c in range(10):
    idx = [i for i, t in enumerate(test_ds.targets[:500]) if t == c][:2]
    for r in range(2):
        axes[r, c].imshow(test_ds[idx[r]][0][0], cmap="gray"); axes[r, c].axis("off")
    axes[0, c].set_title(CLASSES[c], fontsize=9)
plt.suptitle("Fashion-MNIST 클래스별 예시"); plt.tight_layout(); plt.show()



---



## **2-2. 모델 정의 — MLP(기준) vs SmallCNN**

| 모델 | 구조 |
|---|---|
| MLP | 784 → 256 → 10 (픽셀을 한 줄로 편다 → 공간 정보 손실) |
| SmallCNN | [Conv3×3 → BN → ReLU → MaxPool] × 2 → Flatten → Dropout → Linear |

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(),
                                 nn.Linear(28 * 28, 256),
                                 nn.ReLU(),
                                 nn.Linear(256, n_classes))
    def forward(self, x):
        return self.net(x)


class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),   # (N,1,28,28) → (N,32,28,28)
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),                  # → (N,32,14,14)
            nn.Conv2d(32, 64, 3, padding=1),  # → (N,64,14,14)
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),                  # → (N,64,7,7)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                     # → (N, 64*7*7 = 3136)
            nn.Dropout(0.3),
            nn.Linear(64 * 7 * 7, n_classes), # → (N,10)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f"MLP      파라미터 = {count_params(MLP()):>8,}")
print(f"SmallCNN 파라미터 = {count_params(SmallCNN()):>8,}")

### **[실습] SmallCNN 모델의 각 Layer 추적하기**
SmallCNN에서 정의한 각 Layer에서 아래와 같이 주석의 값들이 맞는지 print(x.shape) 로 확인해 봅니다.

In [ ]:
import torch

model = SmallCNN()
model.eval()

# MNIST 이미지 1개: (Batch, Channel, Height, Width)
x = torch.randn(1, 1, 28, 28)

print("입력 Shape:", tuple(x.shape))
print("-" * 55)

with torch.no_grad():

    # 특징 추출기
    for i, layer in enumerate(model.features):
        x = layer(x)
        print(f"features[{i}] {layer.__class__.__name__:15s} → {tuple(x.shape)}")

    # 분류기
    for i, layer in enumerate(model.classifier):
        x = layer(x)
        print(f"classifier[{i}] {layer.__class__.__name__:15s} → {tuple(x.shape)}")



## **2-3. 층별 shape·파라미터 추적** (forward hook)

**register_forward_hook**를 사용하여 `print(x.shape)`를 층마다 넣는 대신, **hook**으로 한 번에 출력합니다.

In [ ]:
def trace_shapes(model, x):
    rows = []
    def hook(mod, inp, out):
        rows.append((mod.__class__.__name__, tuple(inp[0].shape), tuple(out.shape),
                     sum(p.numel() for p in mod.parameters(recurse=False))))

    hs = [m.register_forward_hook(hook) for m in model.modules() if len(list(m.children())) == 0]
    model.eval()
    with torch.no_grad():
        model(x)

    for h in hs: h.remove()
    print(f"{'층':<12}{'입력 모양':<20}{'출력 모양':<20}{'파라미터':>8}")
    for name, i, o, n in rows:
        print(f"{name:<12}{str(i):<20}{str(o):<20}{n:>8,}")

trace_shapes(SmallCNN(), torch.zeros(1, 1, 28, 28))

## **2-4. 학습·평가 함수**

In [ ]:
def run_epoch(model, loader, opt=None):
    """opt가 있으면 학습, 없으면 평가. (평균 손실, 정확도) 반환"""
    train = opt is not None
    model.train(train)
    total, correct, loss_sum = 0, 0, 0.0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = F.cross_entropy(out, y)
            if train:
                opt.zero_grad()      # ① 기울기 초기화
                loss.backward()      # ② 역전파
                opt.step()           # ③ 가중치 갱신
            loss_sum += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            total += len(y)
    return loss_sum / total, correct / total

def fit(model, epochs=3, lr=1e-3):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    hist = []
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, train_loader, opt)
        va_loss, va_acc = run_epoch(model, val_loader)
        sec = time.time() - t0
        hist.append(dict(epoch=ep, tr_loss=tr_loss, va_loss=va_loss, tr_acc=tr_acc, va_acc=va_acc, sec=sec))
        print(f"  epoch {ep}  train_loss {tr_loss:.3f}  val_acc {va_acc:.3f}  ({sec:.1f}초)")
    return hist

## **2-5. MLP vs SmallCNN 비교**: (같은 데이터 · 같은 3에폭으로 공정 비교)
- [주의!] GPU로 설정하고 실행하기

In [ ]:
EPOCHS = 3
results = {}
for name, cls in [("MLP", MLP), ("SmallCNN", SmallCNN)]:
    torch.manual_seed(42)
    model = cls()
    print(f"[{name}] 학습 시작 (파라미터 {count_params(model):,})")
    hist = fit(model, EPOCHS)
    _, te_acc = run_epoch(model, test_loader)
    results[name] = dict(model=model,
                         hist=hist,
                         test_acc=te_acc,
                         params=count_params(model),
                         sec=np.mean([h["sec"] for h in hist]))

print(f"\n{'모델':<10}{'파라미터':>10}{'검증 정확도':>12}{'테스트 정확도':>14}{'에폭당 시간(초)':>16}")
for n, r in results.items():
    print(f"{n:<10}{r['params']:>10,}{r['hist'][-1]['va_acc']:>12.3f}{r['test_acc']:>14.3f}{r['sec']:>16.1f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for n, r in results.items():
    ep = [h["epoch"] for h in r["hist"]]
    axes[0].plot(ep, [h["va_loss"] for h in r["hist"]], "o-", label=n)
    axes[1].plot(ep, [h["va_acc"] for h in r["hist"]], "o-", label=n)
axes[0].set_title("검증 손실"); axes[1].set_title("검증 정확도")
for ax in axes: ax.set_xlabel("epoch"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

cnn = results["SmallCNN"]["model"].eval()
torch.save(cnn.state_dict(), "smallcnn_fashion.pt")   # 과제에서 다시 사용

## **2-6. CNN이 학습한 것 보기** — 필터와 특징맵

In [ ]:
w = cnn.features[0].weight.detach().cpu()[:, 0]          # conv1 필터 32개, (32,3,3)
fig, axes = plt.subplots(2, 16, figsize=(14, 2.2))
for i, ax in enumerate(axes.flat):
    vmax = w[i].abs().max()
    ax.imshow(w[i], cmap="bwr", vmin=-vmax, vmax=vmax); ax.axis("off")
plt.suptitle("conv1이 학습한 3×3 필터 32개 (빨강=양수, 파랑=음수)"); plt.tight_layout(); plt.show()

# 부츠(클래스 9) 한 장에 대한 특징맵
img, lab = next((test_ds[i] for i in range(100) if test_ds[i][1] == 9))
feats = {}
h1 = cnn.features[2].register_forward_hook(lambda m, i, o: feats.__setitem__("conv1", o))   # ReLU 뒤
h2 = cnn.features[6].register_forward_hook(lambda m, i, o: feats.__setitem__("conv2", o))
with torch.no_grad():
    cnn(img[None].to(device))
h1.remove();
h2.remove()

fig, axes = plt.subplots(2, 9, figsize=(14, 3.4))
axes[0, 0].imshow(img[0], cmap="gray")
axes[0, 0].set_title("입력")
axes[1, 0].axis("off")
for k in range(8):
    axes[0, k + 1].imshow(feats["conv1"][0, k].cpu(), cmap="viridis")
    axes[0, k + 1].set_title(f"conv1 #{k}", fontsize=8)
    axes[1, k + 1].imshow(feats["conv2"][0, k].cpu(), cmap="viridis")
    axes[1, k + 1].set_title(f"conv2 #{k}", fontsize=8)
for ax in axes.flat: ax.axis("off")
plt.suptitle("특징맵: conv1(28×28)은 윤곽, conv2(14×14)는 더 추상적인 패턴"); plt.tight_layout(); plt.show()

## **2-7. 오분류 분석 — 혼동행렬**

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

preds, labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        preds.append(cnn(x.to(device)).argmax(1).cpu())
        labels.append(y)
preds, labels = torch.cat(preds).numpy(), torch.cat(labels).numpy()
cm = confusion_matrix(labels, preds)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10))
ax.set_xticklabels(CLASSES, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(10))
ax.set_yticklabels(CLASSES, fontsize=8)
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7, color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xlabel("예측"); ax.set_ylabel("정답"); ax.set_title("SmallCNN 혼동행렬 (테스트 10,000장)")
plt.tight_layout(); plt.show()
print()

# 가장 많이 헷갈린 (정답 → 예측) 쌍 Top 5
off = cm.copy(); np.fill_diagonal(off, 0)
top = np.dstack(np.unravel_index(np.argsort(off.ravel())[::-1][:5], off.shape))[0]
print("가장 많이 헷갈린 쌍:")
for t, p in top:
    print(f"  {CLASSES[t]:>8} → {CLASSES[p]:<8} {off[t, p]}장")
print("\n클래스별 재현율(recall):")
for c in range(10):
    print(f"  {CLASSES[c]:>8}: {cm[c, c] / cm[c].sum():.3f}")

## **2-8. 구조를 바꾸면 무엇이 달라질까?** (선택 실행)

| 번호 |실험 | 확인할 것 |
|---|---|---|
| M1 |  BatchNorm 제거 | 1에폭 검증 정확도 변화 |
| M2 | MaxPool 대신 `stride=2` 합성곱 | 파라미터·정확도 |
| M3 | 마지막을 **전역 평균 풀링(GAP)** + Linear(64→10) | 파라미터가 얼마나 줄어드나? |

`RUN_MISSION = True` 로 바꾸면 세 변형을 각각 1에폭씩 학습합니다 (CPU 약 2분).

In [ ]:
RUN_MISSION = False

def make_variant(kind):
    layers = []
    c_in = 1
    for c_out in [32, 64]:
        if kind == "stride2":
            layers += [nn.Conv2d(c_in, c_out, 3, padding=1, stride=2), nn.BatchNorm2d(c_out), nn.ReLU()]
        else:
            layers += [nn.Conv2d(c_in, c_out, 3, padding=1)]
            if kind != "noBN": layers += [nn.BatchNorm2d(c_out)]
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
        c_in = c_out
    head = ([nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 10)] if kind == "GAP"
            else [nn.Flatten(), nn.Dropout(0.3), nn.Linear(64 * 7 * 7, 10)])
    return nn.Sequential(*layers, *head)

for kind in ["noBN", "stride2", "GAP"]:
    m = make_variant(kind)
    print(f"{kind:<8} 파라미터 {count_params(m):>7,}", end="")
    if RUN_MISSION:
        torch.manual_seed(42)
        h = fit(m, 1)
        # h = fit(m, 3)
        print(f"  → val_acc {h[-1]['va_acc']:.3f}")
    else:
        print()